Import neccessary libraries

In [1]:
import os
import json
import logging
import pandas as pd
import optuna
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)

# 1. THIẾT LẬP LOGGING (Dùng force=True để Jupyter không bị lỗi khi chạy lại cell nhiều lần)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True 
)
logger = logging.getLogger(__name__)

# 2. ĐƯỜNG DẪN DỮ LIỆU
CLEANED_DATA_PATH = Path("../data/processed/cleaned_data.csv")
CONFIG_PATH = Path("../artifacts/best_model_config.json")
TARGET_COLUMN = "churn_status"

e:\Documents\DSEB.NEU.6TH\ML_OPS\MLOpsProject\MLOpsProject\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Netflix_Model_Tuning")
mlflow.sklearn.autolog()

In [3]:
df = pd.read_csv(CLEANED_DATA_PATH)

In [4]:
train_set = df[df['data_split'] == 'train']
test_set = df[df['data_split'] == 'test']

X_train = train_set.drop(columns=['churn_status', 'data_split'])
y_train = train_set['churn_status']

X_test = test_set.drop(columns=['churn_status', 'data_split'])
y_test = test_set['churn_status']

In [5]:
# def objective(trial):
#     # Define hyperparameter search space
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 500), 
#         "max_depth": trial.suggest_int("max_depth", 5, 25),
#         "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
#         "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
#         "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
#         "random_state": 42,
#         "class_weight": "balanced",
#         "n_jobs": -1  # Use all cores for faster training
#     }

#     with mlflow.start_run(nested=True):
#         model = RandomForestClassifier(**params)
#         model.fit(X_train, y_train)
        
#         # Get predictions
#         y_pred = model.predict(X_test)
#         y_proba = model.predict_proba(X_test)[:, 1] # Probability of the positive class

#         # Calculate all required metrics
#         metrics = {
#             "accuracy": accuracy_score(y_test, y_pred),
#             "precision": precision_score(y_test, y_pred, zero_division=0),
#             "recall": recall_score(y_test, y_pred),
#             "f1_score": f1_score(y_test, y_pred),
#             "roc_auc": roc_auc_score(y_test, y_proba)
#         }

#         # Log parameters and all metrics to MLflow
#         mlflow.log_params(params)
#         mlflow.log_metrics(metrics)
        
#         # Optuna maximizes the returned value (Precision)
#         return metrics["precision"]

# # 2. Execute Tuning
# with mlflow.start_run(run_name="RandomForest_Precision_Optimization"):
#     # Create study to maximize Precision
#     study = optuna.create_study(direction="maximize")
#     study.optimize(objective, n_trials=10) 

#     # Log best trial details to parent run
#     mlflow.log_params(study.best_params)
#     mlflow.log_metric("best_precision_score", study.best_value)

#     # Re-train final model with the winner parameters
#     best_model = RandomForestClassifier(**study.best_params, random_state=42)
#     best_model.fit(X_train, y_train)
    
#     # Save the model
#     mlflow.sklearn.log_model(best_model, "best_precision_model")

#     print("-" * 30)
#     print(f"Tuning completed. Best Precision: {study.best_value:.4f}")
#     print(f"Best Parameters: {study.best_params}")

In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # 1. Lấy độ quan trọng của các feature từ model tốt nhất
# importances = best_model.feature_importances_
# feature_names = X_train.columns

# # 2. Tạo DataFrame để dễ sắp xếp
# feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# # 3. In ra Top 5
# print("Top 5 Important Features:")
# print(feature_importance_df.head(5))

# # 4. Vẽ biểu đồ trực quan
# plt.figure(figsize=(10, 6))
# plt.barh(feature_importance_df['Feature'].head(5), feature_importance_df['Importance'].head(5), color='skyblue')
# plt.xlabel('Importance Score')
# plt.title('Top 5 Features causing potential Data Leakage')
# plt.gca().invert_yaxis()
# plt.show()

NameError: name 'best_model' is not defined

In [7]:
# 1. Setup MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Netflix_XGBoost_Tuning")

# Tính toán tỷ lệ mất cân bằng để hỗ trợ XGBoost
# ratio = số mẫu nhãn 0 / số mẫu nhãn 1
ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

def objective(trial):
    # Search space tối ưu cho XGBoost
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        # scale_pos_weight giúp mô hình chú ý hơn vào lớp Churn (1)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, ratio * 1.5),
        "random_state": 42,
        "eval_metric": "logloss"
    }

    with mlflow.start_run(nested=True):
        model = XGBClassifier(**params)
        model.fit(X_train, y_train)
        
        # Dự đoán
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        # Tính toán các chỉ số
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_proba)
        }

        # Log lên MLflow
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        
        # Tối ưu hóa theo Precision như bạn yêu cầu
        return metrics["precision"]

# 2. Chạy Optimization
with mlflow.start_run(run_name="XGBoost_Precision_Focus"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50) # XGBoost cần nhiều trial hơn một chút

    # Log tham số tốt nhất
    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_precision", study.best_value)

    # Huấn luyện model cuối cùng
    best_xgb = XGBClassifier(**study.best_params, random_state=42)
    best_xgb.fit(X_train, y_train)
    
    # Lưu model vào MLflow
    mlflow.xgboost.log_model(best_xgb, "best_xgboost_model")

    print(f"Tuning hoàn tất! Best Precision: {study.best_value:.4f}")

[I 2026-04-05 14:46:02,976] A new study created in memory with name: no-name-6c3ec08e-5da7-49f7-b1ce-46668757c257
[I 2026-04-05 14:46:03,619] Trial 0 finished with value: 0.2353961827646038 and parameters: {'n_estimators': 158, 'max_depth': 7, 'learning_rate': 0.11697201733769275, 'subsample': 0.7623929064445167, 'colsample_bytree': 0.7752407112469396, 'gamma': 1.7829388466901819, 'scale_pos_weight': 3.599722540814296}. Best is trial 0 with value: 0.2353961827646038.


🏃 View run beautiful-stork-512 at: http://127.0.0.1:5000/#/experiments/2/runs/bc1411885e1a4e5fbc183fb1be65801f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


[I 2026-04-05 14:46:04,682] Trial 1 finished with value: 0.2203423967774421 and parameters: {'n_estimators': 387, 'max_depth': 5, 'learning_rate': 0.08940232508623863, 'subsample': 0.685900949158328, 'colsample_bytree': 0.6068932864786825, 'gamma': 0.28788183873345197, 'scale_pos_weight': 6.62463523692627}. Best is trial 0 with value: 0.2353961827646038.


🏃 View run delightful-carp-426 at: http://127.0.0.1:5000/#/experiments/2/runs/abd4cb78f4784c3f86d2ed97b79e8f2a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


[I 2026-04-05 14:46:06,777] Trial 2 finished with value: 0.2600660066006601 and parameters: {'n_estimators': 731, 'max_depth': 9, 'learning_rate': 0.04508476784361892, 'subsample': 0.6375020333166824, 'colsample_bytree': 0.6367303113043505, 'gamma': 3.8172290173582213, 'scale_pos_weight': 4.543622003089011}. Best is trial 2 with value: 0.2600660066006601.


...

🏃 View run beautiful-goose-129 at: http://127.0.0.1:5000/#/experiments/2/runs/cf942398bf4e404d989c3b5dd04581a9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


[I 2026-04-05 14:48:21,498] Trial 49 finished with value: 0.2574329224075417 and parameters: {'n_estimators': 476, 'max_depth': 7, 'learning_rate': 0.020149022976710954, 'subsample': 0.6084595770962635, 'colsample_bytree': 0.5778714937411259, 'gamma': 2.1827558914392275, 'scale_pos_weight': 3.1809935910424625}. Best is trial 39 with value: 0.38095238095238093.


🏃 View run loud-hawk-377 at: http://127.0.0.1:5000/#/experiments/2/runs/0b079007b0c3423aabdaffdb81f29dfc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/04/05 14:48:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Tuning hoàn tất! Best Precision: 0.3810
🏃 View run XGBoost_Precision_Focus at: http://127.0.0.1:5000/#/experiments/2/runs/61557b2b52cf45d1abdd4d0b409247b5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [9]:
import optuna.visualization as vis

# Vẽ biểu đồ tọa độ song song
fig = vis.plot_parallel_coordinate(study)
fig.show()